# US consumer camera/media idea scoring audit

## TL;DR

Recomputes v4.0 Consumer Utility low/base/high opportunity scores and gate-aware desk decisions for four iPhone app candidates. Occasion Reel is the first solo-adjusted product test; MemoryMotion has the highest raw score but materially higher AI operations; RoomCheck remains blocked on an unknown core-feasibility gate.

## Context and method

Inputs are evidence-bounded judgments from four validated Quick Scans dated 2026-07-18. All ideas use the `consumer_utility` archetype, so the same weights apply. Scores are not forecasts or probabilities. Fatal gates and evidence strength override raw score when assigning the desk decision and product order.

In [1]:
from decimal import Decimal, ROUND_HALF_UP

WEIGHTS = {
    'problem_demand': 25,
    'monetization_economics': 15,
    'competitive_wedge': 20,
    'distribution_access': 25,
    'repeat_value_retention': 5,
    'timing': 10,
}

IDEAS = {
    'Occasion Reel': {
        'scores': {
            'problem_demand': (3, 4, 5), 'monetization_economics': (2, 3, 4),
            'competitive_wedge': (2, 3, 4), 'distribution_access': (2, 3, 4),
            'repeat_value_retention': (1, 2, 3), 'timing': (3, 3, 4),
        },
        'evidence': 'C', 'gates': ['clear'] * 4, 'solo_ops': 1,
    },
    'MemoryMotion': {
        'scores': {
            'problem_demand': (3, 4, 5), 'monetization_economics': (2, 3, 4),
            'competitive_wedge': (2, 3, 4), 'distribution_access': (2, 3, 4),
            'repeat_value_retention': (1, 2, 3), 'timing': (4, 4, 5),
        },
        'evidence': 'C', 'gates': ['clear'] * 4, 'solo_ops': 3,
    },
    'Before/After Reel': {
        'scores': {
            'problem_demand': (2, 3, 4), 'monetization_economics': (2, 3, 4),
            'competitive_wedge': (2, 3, 4), 'distribution_access': (2, 3, 4),
            'repeat_value_retention': (2, 3, 4), 'timing': (3, 3, 4),
        },
        'evidence': 'C', 'gates': ['clear'] * 4, 'solo_ops': 1,
    },
    'RoomCheck': {
        'scores': {
            'problem_demand': (3, 4, 4), 'monetization_economics': (2, 3, 4),
            'competitive_wedge': (2, 3, 4), 'distribution_access': (2, 3, 4),
            'repeat_value_retention': (1, 2, 3), 'timing': (3, 3, 4),
        },
        'evidence': 'D', 'gates': ['clear', 'unknown', 'clear', 'clear'], 'solo_ops': 2,
    },
}

def round_half_up(value):
    return int(Decimal(str(value)).quantize(Decimal('1'), rounding=ROUND_HALF_UP))

def opportunity_interval(spec):
    return tuple(round_half_up(sum(WEIGHTS[d] * spec['scores'][d][i] / 5 for d in WEIGHTS)) for i in range(3))

def classification(interval):
    low, _, high = interval
    if high < 40: return 'weak'
    if low >= 60: return 'attractive'
    return 'borderline'

def desk_decision(interval, evidence, gates):
    low, _, high = interval
    if 'blocked' in gates: return 'ELENDIR'
    if 'unknown' in gates or evidence == 'D': return 'BELIRSIZ'
    if high < 40: return 'ELENDIR'
    if low >= 60: return 'TEST_ONCELIKLI'
    return 'TESTE_DEGER'

expected = {
    'Occasion Reel': ((46, 64, 84), 'TESTE_DEGER'),
    'MemoryMotion': ((48, 66, 86), 'TESTE_DEGER'),
    'Before/After Reel': ((42, 60, 80), 'TESTE_DEGER'),
    'RoomCheck': ((46, 64, 79), 'BELIRSIZ'),
}

rows = []
for name, spec in IDEAS.items():
    interval = opportunity_interval(spec)
    decision = desk_decision(interval, spec['evidence'], spec['gates'])
    assert (interval, decision) == expected[name], (name, interval, decision)
    assert classification(interval) == 'borderline'
    rows.append((name, *interval, spec['evidence'], decision, spec['solo_ops']))

print('idea | low | base | high | evidence | decision | solo_ops')
print('--- | --- | --- | --- | --- | --- | ---')
for row in rows:
    print(' | '.join(map(str, row)))

raw_order = sorted(IDEAS, key=lambda n: opportunity_interval(IDEAS[n])[1], reverse=True)
product_order = ['Occasion Reel', 'MemoryMotion', 'Before/After Reel', 'RoomCheck']
assert raw_order[0] == 'MemoryMotion'
assert product_order[0] == 'Occasion Reel'
assert desk_decision(opportunity_interval(IDEAS['RoomCheck']), 'D', IDEAS['RoomCheck']['gates']) == 'BELIRSIZ'
print('\nRaw base-score order:', ' > '.join(raw_order))
print('Gate/solo-adjusted product order:', ' > '.join(product_order))
print('PASS: arithmetic, classification, decisions, and founder-fit ordering assertions')


idea | low | base | high | evidence | decision | solo_ops
--- | --- | --- | --- | --- | --- | ---
Occasion Reel | 46 | 64 | 84 | C | TESTE_DEGER | 1
MemoryMotion | 48 | 66 | 86 | C | TESTE_DEGER | 3
Before/After Reel | 42 | 60 | 80 | C | TESTE_DEGER | 1
RoomCheck | 46 | 64 | 79 | D | BELIRSIZ | 2

Raw base-score order: MemoryMotion > Occasion Reel > RoomCheck > Before/After Reel
Gate/solo-adjusted product order: Occasion Reel > MemoryMotion > Before/After Reel > RoomCheck
PASS: arithmetic, classification, decisions, and founder-fit ordering assertions


## Takeaways

- MemoryMotion leads raw score because current AI-video timing is stronger, but provider cost, retries, privacy, refunds and support make it a heavier solo business.
- Occasion Reel is the first product-behavior test because all gates are clear, the workflow can stay local and the missed insight is testable without a full backend.
- RoomCheck cannot advance on score: the unknown core-feasibility gate and overall evidence D force `BELIRSIZ`.
- No desk score authorizes a build; each candidate must pass its registered behavior or technical test.